In [ ]:
# %%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {install_numpy} \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes transformers==4.55.2 \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.2/21.2 MB 74.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load model using Unsloth FastLanguageModel for function calling
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Increased for function calling complexity
dtype = torch.bfloat16
model_name = "meta-llama/Llama-2-7b-chat-hf"  # Changed to Llama 2 for better function calling

# Load model and tokenizer with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    dtype=dtype,
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # Use 4-bit quantization to save memory
    full_finetuning=False,
)

print("✅ Model and tokenizer loaded successfully!")
print(f"Model: {model_name}")
print(f"Max sequence length: {max_seq_length}")
print(f"Using dtype: {dtype}")


In [ ]:
# ============================================================================
# UTILITY FUNCTIONS FOR FUNCTION CALLING FINE-TUNING
# ============================================================================

import json
import re

def extract_json_from_response(response: str) -> dict:
    """
    Extract JSON function call from model response
    
    Args:
        response: Model generated text
    
    Returns:
        Parsed JSON dict or None if no valid JSON found
    """
    # Try to find JSON block
    json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    return None


def validate_function_call(response: str) -> bool:
    """
    Check if response contains valid function call JSON
    
    Returns:
        True if valid JSON with 'name' and 'parameters' fields
    """
    try:
        data = extract_json_from_response(response)
        if data and "name" in data and "parameters" in data:
            return True
    except:
        pass
    return False


def handle_function_calling(query: str, tokenizer, model):
    """
    Handle function calling for market research tasks
    
    Args:
        query: User input with tool definitions
        tokenizer: Tokenizer for model
        model: Fine-tuned model for function calling
    
    Returns:
        dict: Parsed function call or error
    """
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a market research analyst. You must respond with valid JSON function calls."
            },
            {
                "role": "user",
                "content": query
            }
        ]
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Tokenize
        model_inputs = tokenizer([text], return_tensors="pt")
        
        if torch.cuda.is_available():
            model_inputs = model_inputs.to('cuda')
        
        # Generate response
        with torch.no_grad():
            generated_ids = model.generate(
                model_inputs.input_ids,
                max_new_tokens=500,
                temperature=0.3,  # Low temperature for JSON precision
                top_p=0.95,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        # Decode response
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract and validate JSON
        json_data = extract_json_from_response(response)
        is_valid = json_data is not None and "name" in json_data and "parameters" in json_data
        
        return {
            "success": is_valid,
            "response": response,
            "json_data": json_data,
            "valid_function_call": is_valid
        }
    
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "valid_function_call": False
        }


def test_function_calling(tokenizer, model, test_cases=None):
    """
    Test function calling with sample test cases
    
    Args:
        tokenizer: Tokenizer
        model: Fine-tuned model
        test_cases: List of test prompts (optional)
    """
    if test_cases is None:
        test_cases = [
            # Intent classification test
            'Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.\n\n{"name": "classify_intent", "description": "Classify user prompt", "parameters": {"type": "object", "properties": {"intent": {"type": "string", "enum": ["chat", "knowledge", "research"]}, "response": {"type": "string"}}, "required": ["intent", "response"]}}\n\nXin chào! Bạn khỏe không?',
            
            # Planning test
            'Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.\n\n{"name": "create_research_plan", "parameters": {"type": "object", "properties": {"research_questions": {"type": "array", "items": {"type": "string"}}, "search_steps": {"type": "array", "items": {"type": "object", "properties": {"query": {"type": "string"}, "focus": {"type": "string"}}}}}, "required": ["research_questions", "search_steps"]}}\n\nPhân tích thị trường cà phê specialty ở Việt Nam',
        ]
    
    print("\n" + "="*70)
    print("🧪 TESTING FUNCTION CALLING CAPABILITY")
    print("="*70 + "\n")
    
    success_count = 0
    for i, test_prompt in enumerate(test_cases, 1):
        print(f"Test Case {i}:")
        print("-" * 70)
        
        result = handle_function_calling(test_prompt, tokenizer, model)
        
        if result["valid_function_call"]:
            success_count += 1
            print("✅ VALID FUNCTION CALL")
            print(f"Function: {result['json_data'].get('name')}")
            print(f"Parameters: {json.dumps(result['json_data'].get('parameters'), indent=2, ensure_ascii=False)}")
        else:
            print("❌ INVALID FUNCTION CALL")
            print(f"Response: {result['response'][:200]}...")
        
        print()
    
    print("="*70)
    print(f"✅ SUCCESS RATE: {success_count}/{len(test_cases)} ({100*success_count//len(test_cases)}%)")
    print("="*70 + "\n")
    
    return success_count / len(test_cases)


print("✅ Function calling utilities loaded successfully!")


Utility functions loaded successfully!


In [ ]:
# ============================================================================
# SETUP LoRA FOR FUNCTION CALLING + LOAD TRAINING DATA
# ============================================================================

# Setup LoRA optimized for function calling
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Higher rank for better function call accuracy
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # Higher alpha for stronger updates
    lora_dropout=0.05,  # Small dropout for stability
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA setup completed (optimized for function calling)!")

# Load training data
import json
from datasets import load_dataset

print("\n📥 Loading training data...")

# Load from training_data.json
with open("/content/drive/MyDrive/backend/training_data.json", "r", encoding="utf-8") as f:
    raw_training_data = json.load(f)

print(f"✅ Loaded {len(raw_training_data)} training examples")

# Convert to messages format (handle both old and new format)
training_examples = []
for example in raw_training_data:
    # New format: messages field directly
    if "messages" in example:
        training_examples.append({
            "messages": example["messages"]
        })
    # Old format: conversation field (backward compatibility)
    elif "conversation" in example:
        training_examples.append({
            "messages": example["conversation"]
        })

print(f"✅ Converted to messages format: {len(training_examples)} examples")

if len(training_examples) == 0:
    print("❌ WARNING: No training examples found! Check data format.")
    print(f"   Sample keys in first example: {list(raw_training_data[0].keys()) if raw_training_data else 'EMPTY'}")

# Create dataset
from datasets import Dataset
dataset = Dataset.from_dict({
    "messages": [ex["messages"] for ex in training_examples]
})

print(f"✅ Dataset created successfully")
print(f"Example 1 structure: {len(dataset[0]['messages'])} messages")
for i, msg in enumerate(dataset[0]['messages']):
    role = msg.get('role', 'unknown')
    if role == 'tool_calls':
        print(f"  - Message {i}: tool_calls found ✅")
    else:
        content_preview = msg.get('content', '')[:50] if msg.get('content') else '[tool_calls]'
        print(f"  - Message {i}: {role} - {content_preview}...")

# Format using apply_chat_template for function calling
def format_function_calling(examples):
    """Format examples using Llama chat template for function calling"""
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

print("\n🔄 Formatting dataset for function calling...")
dataset = dataset.map(format_function_calling, batched=True, batch_size=8)

print("✅ Dataset formatted successfully!")
print(f"Sample formatted text length: {len(dataset[0]['text'])} characters")
print(f"First 300 chars: {dataset[0]['text'][:300]}...")

# Setup training configuration
from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    num_train_epochs=3,  # More epochs for small dataset
    learning_rate=1e-4,  # Lower learning rate for precision
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",  # Better for small datasets
    seed=3407,
    output_dir="./llama_function_calling_finetune",
    report_to="none",
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    max_grad_norm=1.0,  # Gradient clipping for stability
)

print("\n✅ Training configuration ready!")
print(f"   - Batch size: {training_config.per_device_train_batch_size}")
print(f"   - Learning rate: {training_config.learning_rate}")
print(f"   - Epochs: {training_config.num_train_epochs}")
print(f"   - Dataset size: {len(dataset)}")

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_config,
)

print("✅ SFTTrainer initialized successfully!")
print("\n🚀 Ready to start training...")

Unsloth 2025.10.1 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


LoRA setup completed with Unsloth!


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1888 [00:00<?, ? examples/s]

Dataset loaded and formatted successfully with 1888 samples
Sample formatted text:
<|system|>
You are Mia, a chatty AI with a friendly, laid-back vibe. Clarify vague questions briefly.</s>
<|user|>
What’s a cool place to go?</s>
<|assistant|>
Cool place for what? Cafe, beach, or hik...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

SFTTrainer setup completed successfully!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 6,307,840 of 1,106,356,224 (0.57% trained)


Step,Training Loss
1,2.173900
2,2.036200
3,2.165200
4,1.925600
5,2.030100
6,1.915400
7,1.786800
8,1.619800
9,1.617300
10,1.599600


Training completed successfully!
Model saved successfully!


In [ ]:
# ============================================================================
# TRAINING FOR FUNCTION CALLING
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING FUNCTION CALLING FINE-TUNING")
print("="*70 + "\n")

# Start training
trainer_stats = trainer.train()

print("\n" + "="*70)
print("✅ TRAINING COMPLETED")
print("="*70)
print(f"Final loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.training_steps} steps")

# Save the fine-tuned model
model.save_pretrained("./llama_function_calling_finetune")
tokenizer.save_pretrained("./llama_function_calling_finetune")
print("\n✅ Model saved to ./llama_function_calling_finetune")

# Clear memory
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ GPU memory cleared")


In [ ]:
# ============================================================================
# EVALUATE FUNCTION CALLING CAPABILITY
# ============================================================================

print("\n" + "="*70)
print("📊 EVALUATING FUNCTION CALLING ACCURACY")
print("="*70 + "\n")

# Test cases covering different function types
test_cases = [
    # Test 1: Intent Classification
    'Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.\n\n{"name": "classify_intent", "description": "Classify user prompt into chat, knowledge, or research", "parameters": {"type": "object", "properties": {"intent": {"type": "string", "enum": ["chat", "knowledge", "research"]}, "response": {"type": "string"}, "reasoning": {"type": "string"}}, "required": ["intent", "response", "reasoning"]}}\n\nXin chào! Bạn khỏe không?',
    
    # Test 2: Research Planning
    'Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.\n\n{"name": "create_research_plan", "description": "Create research questions and search steps", "parameters": {"type": "object", "properties": {"research_questions": {"type": "array", "items": {"type": "string"}}, "hypotheses": {"type": "array", "items": {"type": "string"}}, "search_steps": {"type": "array", "items": {"type": "object", "properties": {"query": {"type": "string"}, "focus": {"type": "string"}}, "required": ["query", "focus"]}}}, "required": ["research_questions", "hypotheses", "search_steps"]}}\n\nPhân tích thị trường cà phê specialty ở Việt Nam',
    
    # Test 3: ReAct Decision
    'Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.\n\n{"name": "decide_action", "description": "Decide next action in ReAct loop", "parameters": {"type": "object", "properties": {"action": {"type": "string", "enum": ["search", "refine_query", "summarize"]}, "query": {"type": "string"}, "reasoning": {"type": "string"}}, "required": ["action", "query", "reasoning"]}}\n\nNhiệm vụ hiện tại: Tìm hiểu quy mô thị trường cà phê\nSố bằng chứng đã có: 5',
    
    # Test 4: SWOT Analysis
    'Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.\n\n{"name": "generate_swot", "description": "Generate SWOT analysis", "parameters": {"type": "object", "properties": {"strengths": {"type": "array", "items": {"type": "string"}}, "weaknesses": {"type": "array", "items": {"type": "string"}}, "opportunities": {"type": "array", "items": {"type": "string"}}, "threats": {"type": "array", "items": {"type": "string"}}}, "required": ["strengths", "weaknesses", "opportunities", "threats"]}}\n\nPhân tích SWOT cho startup cà phê specialty',
]

# Run tests
success_rate = test_function_calling(tokenizer, model, test_cases)

print("\n📈 RESULTS SUMMARY:")
print("="*70)
if success_rate >= 0.75:
    print(f"✅ EXCELLENT! Success rate: {success_rate:.1%}")
    print("   Model is ready for production use!")
elif success_rate >= 0.5:
    print(f"⚠️  GOOD. Success rate: {success_rate:.1%}")
    print("   Model needs more training for higher accuracy.")
else:
    print(f"❌ POOR. Success rate: {success_rate:.1%}")
    print("   Model needs significant improvement.")
print("="*70 + "\n")

# Detailed analysis
print("💡 RECOMMENDATIONS:")
if success_rate < 0.75:
    print("   1. Run more training epochs (increase num_train_epochs)")
    print("   2. Add more diverse training examples")
    print("   3. Reduce learning rate for finer tuning")
    print("   4. Increase LoRA rank for better expressiveness")
else:
    print("   ✅ Model is performing well!")
    print("   ✅ Ready to integrate with MarketMind-AI pipelines")
    print("   ✅ Consider A/B testing with production data")
print()


In [ ]:
# Save fine-tuned model to Google Drive
import shutil
import os

print("📤 Saving model to Google Drive...")

source_dir = "./llama_function_calling_finetune"
dest_dir = "/content/drive/MyDrive/backend/models/llama_function_calling_finetune"

# Create destination directory
os.makedirs(dest_dir, exist_ok=True)

# Copy all files
for item in os.listdir(source_dir):
    src_path = os.path.join(source_dir, item)
    dst_path = os.path.join(dest_dir, item)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

print(f"✅ Model saved to: {dest_dir}")
print(f"   Files: {os.listdir(dest_dir)}")


cp: cannot stat '/content/tiny_llama_finetune/checkpoint-100': No such file or directory


In [ ]:
# ============================================================================
# DEMO: USING FINE-TUNED MODEL FOR FUNCTION CALLING
# ============================================================================

print("\n" + "="*70)
print("🎯 DEMO: REAL-WORLD FUNCTION CALLING TEST")
print("="*70 + "\n")

# Example 1: Market Research Intent Classification
print("Example 1: Intent Classification for Market Research")
print("-" * 70)

query1 = '''Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{
    "name": "classify_intent",
    "description": "Classify user prompt into one of three intents: chat, knowledge, or research",
    "parameters": {
        "type": "object",
        "properties": {
            "intent": {
                "type": "string",
                "enum": ["chat", "knowledge", "research"],
                "description": "The classified intent"
            },
            "response": {
                "type": "string",
                "description": "Provide a friendly, helpful response to the user"
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning for the classification"
            }
        },
        "required": ["intent", "response", "reasoning"]
    }
}

Tôi muốn phân tích thị trường ngành công nghiệp ô tô điện tại Việt Nam, tập trung vào khách hàng trẻ tuổi từ 25-40 tuổi ở thành phố.'''

result1 = handle_function_calling(query1, tokenizer, model)
print(f"Valid JSON: {result1['valid_function_call']}")
if result1['json_data']:
    print(f"Function: {result1['json_data'].get('name')}")
    print(f"Intent: {result1['json_data']['parameters'].get('intent')}")
print()

# Example 2: Research Planning
print("Example 2: Research Planning with Multiple Steps")
print("-" * 70)

query2 = '''Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{
    "name": "create_research_plan",
    "description": "Create research questions, hypotheses, and search steps for market analysis",
    "parameters": {
        "type": "object",
        "properties": {
            "research_questions": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Key research questions"
            },
            "hypotheses": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Initial hypotheses to test"
            },
            "search_steps": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string"},
                        "focus": {"type": "string"}
                    },
                    "required": ["query", "focus"]
                },
                "description": "Ordered list of search queries with focus area"
            }
        },
        "required": ["research_questions", "hypotheses", "search_steps"]
    }
}

Input:
- ban_chat_san_pham: Dịch vụ giao hàng nhanh
- khach_hang_muc_tieu: Dân văn phòng bận rộn
- gia_tri_cot_loi: Nhanh chóng, đáng tin cây
- gia_ca_chinh_sach: Giá hợp lý so với competitor'''

result2 = handle_function_calling(query2, tokenizer, model)
print(f"Valid JSON: {result2['valid_function_call']}")
if result2['json_data']:
    print(f"Function: {result2['json_data'].get('name')}")
    params = result2['json_data'].get('parameters', {})
    if 'research_questions' in params:
        print(f"Number of research questions: {len(params['research_questions'])}")
        print(f"Number of search steps: {len(params.get('search_steps', []))}")
print()

print("="*70)
print("✅ Demo completed successfully!")
print("="*70 + "\n")
